# Bar chart generator


In [161]:
from __future__ import annotations

import importlib
import json
import re
import shutil
import subprocess
import sys
import tempfile
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import colors as mcolors

# Groq
# import os
# from groq import Groq

# UNCOMMENT FOR OLLAMA
import ollama

# UNCOMMENT FOR GROQ
# os.environ["GROQ_API_KEY"] = "your-key-here"

# UNCOMMENT FOR GROQ
# _groq_client = Groq()  # reads GROQ_API_KEY from env


def _optional_import(module_name):
    try:
        return importlib.import_module(module_name)
    except Exception:
        return None


def install_package(pip_name: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


def ensure_module(module_name: str, pip_name: str | None = None):
    module = _optional_import(module_name)
    if module is not None:
        return module
    install_package(pip_name or module_name)
    return importlib.import_module(module_name)


sns = _optional_import("seaborn")
alt = _optional_import("altair")
go  = _optional_import("plotly.graph_objects")

try:
    from IPython.display import display
except Exception:
    display = None

In [162]:
# ollama check 

import ollama

response = ollama.chat(
    model="llama3.2",
    messages=[{"role": "user", "content": "Say hello."}],
)
print(response["message"]["content"])

Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [163]:
from pathlib import Path

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

DATA_TRAIN = Path(r"C:\Users\Michelle\I2R\data\train")

# Normal generated outputs
BAR_OUT_ROOT = Path(r"C:\Users\Michelle\OneDrive - TU Eindhoven\I2R_dataset\barplots")

# Testing outputs
BAR_TEST_OUT_ROOT = PROJECT / "testing" / "bar_testing"

BAR_OUT_ROOT.mkdir(parents=True, exist_ok=True)
BAR_TEST_OUT_ROOT.mkdir(parents=True, exist_ok=True)

CLEAR_OUTPUT = False
LIBRARIES = ["altair", "matplotlib", "seaborn", "plotly"]
SUBDIRS = ["images", "tables", "metadata"]

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


# Data


In [164]:
# ── Dataset registry ──────────────────────────────────────────────────────────
DATASET_REGISTRY = {
    "warehouse_retail": {
        "path":         DATA_TRAIN / "Warehouse_and_Retail_Sales.csv",
        "numeric_cols": ["RETAIL SALES", "WAREHOUSE SALES", "RETAIL TRANSFERS"],
        "group_cols":   ["ITEM TYPE", "SUPPLIER"],
        "date_col":     "date",
        "agg_cols":     ["YEAR", "MONTH"],
        "loader":       "load_warehouse_retail",
    },
    "london_borough_sector_jobs": {
        "path":         DATA_TRAIN / "london_borough_sector_jobs.csv",
        "numeric_cols": ["EMPLOYEE_JOBS"],
        "group_cols":   ["SECTOR", "BOROUGH"],
        "date_col":     None,
        "agg_cols":     ["YEAR"],
        "loader":       "load_london_borough_sector_jobs",
    },
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [165]:
def load_warehouse_retail(df: pd.DataFrame) -> pd.DataFrame:
    df["date"] = pd.to_datetime(
        dict(year=df["YEAR"].astype("Int64"), month=df["MONTH"].astype("Int64"), day=1),
        errors="coerce",
    )
    return df


def load_london_borough_sector_jobs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean London Datastore borough-by-sector employee jobs data.
    Handles CSV formats where the real header row contains year columns.
    """
    import re

    df = df.copy()
    df = df.dropna(axis=1, how="all")
    df = df.dropna(axis=0, how="all").reset_index(drop=True)

    def is_year_like(value) -> bool:
        s = str(value).strip()
        s = re.sub(r"\.0$", "", s)
        return bool(re.fullmatch(r"(19|20)\d{2}", s))

    current_year_cols = [c for c in df.columns if is_year_like(c)]
    if len(current_year_cols) < 5:
        header_row_idx = None
        best_year_count = 0
        for i in range(min(len(df), 30)):
            row_values = df.iloc[i].tolist()
            year_count = sum(is_year_like(v) for v in row_values)
            if year_count > best_year_count:
                best_year_count = year_count
                header_row_idx = i
        if header_row_idx is not None and best_year_count >= 5:
            new_cols = df.iloc[header_row_idx].tolist()
            df = df.iloc[header_row_idx + 1:].reset_index(drop=True)
            df.columns = [str(c) for c in new_cols]

    year_cols = [c for c in df.columns if is_year_like(c)]
    id_cols   = [c for c in df.columns if not is_year_like(c)]

    if not year_cols or not id_cols:
        raise ValueError("Could not identify year columns in london dataset.")

    df_long = df.melt(id_vars=id_cols, value_vars=year_cols,
                      var_name="YEAR", value_name="EMPLOYEE_JOBS")
    df_long["YEAR"] = pd.to_numeric(df_long["YEAR"].str.replace(r"\.0$", "", regex=True),
                                    errors="coerce")
    df_long["EMPLOYEE_JOBS"] = pd.to_numeric(df_long["EMPLOYEE_JOBS"], errors="coerce")
    df_long = df_long.dropna(subset=["YEAR", "EMPLOYEE_JOBS"])
    df_long = df_long[df_long["EMPLOYEE_JOBS"] > 0]

    col_map = {}
    for c in id_cols:
        cl = str(c).strip().upper()
        if "BOROUGH" in cl or "AREA" in cl or "DISTRICT" in cl:
            col_map[c] = "BOROUGH"
        elif "SECTOR" in cl or "INDUSTRY" in cl or "TYPE" in cl:
            col_map[c] = "SECTOR"
    df_long = df_long.rename(columns=col_map)

    if "BOROUGH" not in df_long.columns:
        remaining = [c for c in id_cols if c not in col_map]
        if remaining:
            df_long = df_long.rename(columns={remaining[0]: "BOROUGH"})
    if "SECTOR" not in df_long.columns:
        remaining = [c for c in id_cols if c not in col_map] if col_map else id_cols
        if remaining:
            df_long = df_long.rename(columns={remaining[0]: "SECTOR"})

    return df_long.reset_index(drop=True)


def get_loader(name: str):
    loaders = {
        "load_warehouse_retail":           load_warehouse_retail,
        "load_london_borough_sector_jobs": load_london_borough_sector_jobs,
    }
    return loaders.get(name)


def load_dataset(dataset_name: str) -> pd.DataFrame | None:
    spec = DATASET_REGISTRY.get(dataset_name)
    if spec is None:
        raise KeyError(f"Unknown dataset: {dataset_name}")
    path = spec["path"]
    if not path.exists():
        print(f"Dataset not found: {path}")
        return None
    df = pd.read_csv(path)
    if spec["loader"]:
        loader_fn = get_loader(spec["loader"])
        if loader_fn:
            df = loader_fn(df)
    return df


DATASETS = {}
for ds_name, ds_spec in DATASET_REGISTRY.items():
    ds = load_dataset(ds_name)
    if ds is not None:
        DATASETS[ds_name] = ds
        print(f"Loaded '{ds_name}': {len(ds)} rows, cols: {list(ds.columns)}")

DEFAULT_DATASET = next(iter(DATASETS)) if DATASETS else None
print(f"Default dataset: {DEFAULT_DATASET}")

Loaded 'warehouse_retail': 307645 rows, cols: ['YEAR', 'MONTH', 'SUPPLIER', 'ITEM CODE', 'ITEM DESCRIPTION', 'ITEM TYPE', 'RETAIL SALES', 'RETAIL TRANSFERS', 'WAREHOUSE SALES', 'date']
Loaded 'london_borough_sector_jobs': 1567 rows, cols: ['BOROUGH', 'SECTOR', 'YEAR', 'EMPLOYEE_JOBS']
Default dataset: warehouse_retail


## 1. Sampling weights

Weights are derived directly from observed frequencies in the reference dataset.


In [166]:
# SAMPLING_WEIGHTS = {
#     # is_vertical: 0=horizontal bar, 1=vertical bar
#     "is_vertical":           {False: 0.26, True: 0.74},

#     # title_present: whether a title is shown
#     "title_present":         {False: 0.39, True: 0.61},

#     # title_location: alignment of title (none only when title_present=False)
#     "title_location":        {"none": 0.39, "center": 0.48, "left": 0.13},

#     # title_color: black vs dark gray
#     "title_color":           {"gray": 0.64, "black": 0.36},

#     # title_size: small (~10-12pt) vs average (~12-18pt)
#     "title_size":            {"small": 0.51, "average": 0.49},

#     # title_size_large: overrides title_size to large (~20-24pt)
#     "title_size_large":      {False: 0.88, True: 0.12},

#     # subtitle_present
#     "subtitle_present":      {False: 0.96, True: 0.04},

#     # outline_chart: border around the entire chart area
#     "outline_chart":         {False: 0.90, True: 0.10},

#     # legend_location: 'no legend' or a position string
#     # middle-right(3) → right, top-center(1) → top, bottom-center(1) → bottom
#     "legend_location":       {"no legend": 0.91, "right": 0.03, "top": 0.01,
#                               "bottom": 0.01, "top-left": 0.01, "top-right": 0.03},

#     # legend_title: whether legend has a title label
#     "legend_title":          {False: 0.99, True: 0.01},

#     # legend_title_color: whether legend title is colored (always False in data)
#     "legend_title_color":    {False: 1.00},

#     # legend_outline: border around legend box
#     "legend_outline":        {False: 0.98, True: 0.02},

#     # gridlines: whether any gridlines are shown
#     "gridlines":             {False: 0.48, True: 0.52},

#     # gridlines_both: horizontal AND vertical gridlines
#     "gridlines_both":        {False: 0.87, True: 0.13},

#     # x_labels_orientation: visibility of x-axis tick labels
#     "x_labels_orientation":  {"hidden": 0.08, "horizontal": 0.92},

#     # x_labels_diagonal: x labels at 45 degrees
#     "x_labels_diagonal":     {False: 0.96, True: 0.04},

#     # x_labels_vertical: x labels at 90 degrees
#     "x_labels_vertical":     {False: 0.99, True: 0.01},

#     # y_labels_orientation: visibility of y-axis tick labels
#     "y_labels_orientation":  {"hidden": 0.02, "horizontal": 0.98},

#     # y_labels_diagonal: y labels at 45 degrees (always False in data)
#     "y_labels_diagonal":     {False: 1.00},

#     # y_labels_vertical: y labels at 90 degrees
#     "y_labels_vertical":     {False: 0.99, True: 0.01},

#     # labels_colored: axis tick labels rendered in a non-black color
#     "labels_colored":        {False: 0.85, True: 0.15},

#     # direct_labeling: value labels on bars
#     #   none=no labels, direct_no_axis=labels replace axis, direct_with_axis=labels + axis
#     "direct_labeling":       {"none": 0.85, "direct_no_axis": 0.02, "direct_with_axis": 0.13},

#     # direct_label_position: where the value label sits on the bar
#     "direct_label_position": {"base": 0.85, "outside": 0.15},

#     # aspect_ratio: overall chart shape
#     "aspect_ratio":          {"square": 0.09, "horizontal": 0.89, "vertical": 0.02},

#     # sample_bin: approximate number of bars (mapped to a concrete n_bars value)
#     "sample_bin":            {8: 0.90, 15: 0.05, 25: 0.02, 35: 0.02, 45: 0.01},

#     # colored_bars: multicolor vs single-color bars
#     "colored_bars":          {False: 0.03, True: 0.97},

#     # # palette_type: which colour palette to use when colored_bars=True
#     # "palette_type": {
#     #     0:  0.04,
#     #     1:  0.03,
#     #     2:  0.29,
#     #     3:  0.05,
#     #     4:  0.15,
#     #     5:  0.10,
#     #     6:  0.01,
#     #     7:  0.09,
#     #     8:  0.02,
#     #     9:  0.02,
#     #     10: 0.03,
#     #     11: 0.03,
#     #     13: 0.07,
#     #     14: 0.02,
#     #     15: 0.02,
#     #     16: 0.01,
#     #     17: 0.01,
#     #     18: 0.01,
#     # },

#     "palette_type": {
#     2:  0.20,   # bright
#     3:  0.07,   # orange
#     4:  0.05,   # dark blue (kept but low)
#     5:  0.08,   # grey shades
#     6:  0.08,   # bright blue
#     7:  0.08,   # light multicolor
#     10: 0.08,   # RGB
#     11: 0.08,   # red, blue
#     12: 0.05,   # muted multicolor
#     13: 0.08,   # blue, red, orange
#     14: 0.05,   # blue shades
#     15: 0.07,   # red, orange, green
#     17: 0.07,   # green, blue
#     18: 0.05,   # fallback bright
# },
#     # rounded_corners: bars have rounded top corners
#     "rounded_corners":       {False: 0.91, True: 0.09},

#     # outline_bars: bars have a black border stroke
#     "outline_bars":          {False: 0.73, True: 0.27},

#     # background_non_white: non-white (light gray) plot background
#     # background: white or one of several non-white options
#     "background":            {"white": 0.97, "light_gray": 0.01, "light_color": 0.01, "dark_gray": 0.005, "dark": 0.005},

#     # ticks: which axes show tick marks
#     "ticks":                 {"none": 0.48, "x": 0.08, "both": 0.21, "y": 0.23},
# }





In [167]:
SAMPLING_WEIGHTS = {
    "is_vertical":           {False: 0.26, True: 0.74},
    "title_present":         {False: 0.39, True: 0.61},
    "title_location":        {"none": 0.39, "center": 0.48, "left": 0.13},
    "title_color":           {"gray": 0.64, "black": 0.36},
    "title_size":            {"small": 0.51, "average": 0.49},
    "title_size_large":      {False: 0.88, True: 0.12},
    "subtitle_present":      {False: 0.96, True: 0.04},
    "outline_chart":         {False: 0.90, True: 0.10},
    "legend_location":       {"no legend": 0.91, "right": 0.03, "top": 0.01,
                              "bottom": 0.01, "top-left": 0.01, "top-right": 0.03},
    "legend_title":          {False: 0.99, True: 0.01},
    "legend_title_color":    {False: 1.00},
    "legend_outline":        {False: 0.98, True: 0.02},
    "gridlines":             {False: 0.48, True: 0.52},
    "gridlines_both":        {False: 0.87, True: 0.13},
    "x_labels_orientation":  {"hidden": 0.08, "horizontal": 0.92},
    "x_labels_diagonal":     {False: 0.96, True: 0.04},
    "x_labels_vertical":     {False: 0.99, True: 0.01},
    "y_labels_orientation":  {"hidden": 0.02, "horizontal": 0.98},
    "y_labels_diagonal":     {False: 1.00},
    "y_labels_vertical":     {False: 0.99, True: 0.01},
    "labels_colored":        {False: 0.85, True: 0.15},
    "direct_labeling":       {"none": 0.85, "direct_no_axis": 0.02, "direct_with_axis": 0.13},
    "direct_label_position": {"base": 0.85, "outside": 0.15},
    "aspect_ratio":          {"square": 0.09, "horizontal": 0.89, "vertical": 0.02},
    "sample_bin":            {8: 0.90, 15: 0.05, 25: 0.02, 35: 0.02, 45: 0.01},
    "colored_bars":          {False: 0.03, True: 0.97},
    "rounded_corners":       {False: 0.91, True: 0.09},
    "outline_bars":          {False: 0.73, True: 0.27},
    "background":            {"white": 0.97, "light_gray": 0.01, "light_color": 0.01, "dark_gray": 0.005, "dark": 0.005},
    "ticks":                 {"none": 0.48, "x": 0.08, "both": 0.21, "y": 0.23},
}

In [168]:
TITLE_COLOR_PALETTES = {
    "black": ["#000000", "#1a1a1a","#212121"],
    "gray":  ["#555555", "#6b6b6b", "#888888", "#9e9e9e"],
}

TITLE_SIZE_PALETTES = {
    "small":   [9, 10, 11, 12],
    "average": [13, 14, 15, 16, 17, 18],
    "large":   [20, 21, 22, 23, 24],
}

# ── Palette base colors (RGB tuples) — jitter applied at render time ──────────
# Mirrors the line generator's PALETTE_BASES for consistent contrast-aware sampling.
PALETTE_BASES = {
    0:  [(0,   0,   0  )] * 6,                                          # black
    1:  [(30,  30,  30 ), (60,  60,  60 ), (90,  90,  90 ),
         (40,  50,  60 ), (50,  40,  70 ), (40,  70,  50 )],            # dark
    2:  [(31,  119, 180), (255, 127, 14 ), (44,  160, 44 ),
         (214, 39,  40 ), (148, 103, 189), (23,  190, 207)],            # bright
    3:  [(255, 140, 0  ), (230, 100, 0  ), (200, 80,  0  ),
         (255, 165, 50 ), (210, 120, 20 ), (240, 150, 30 )],            # orange
    4:  [(0,   30,  100), (0,   50,  130), (0,   70,  160),
         (20,  60,  140), (10,  40,  120), (30,  80,  150)],            # dark blue
    5:  [(50,  50,  50 ), (100, 100, 100), (150, 150, 150),
         (180, 180, 180), (200, 200, 200), (80,  80,  80 )],            # grey shades
    6:  [(0,   100, 200), (30,  130, 220), (60,  160, 240),
         (0,   80,  180), (20,  110, 210), (50,  140, 230)],            # bright blue
    7:  [(180, 220, 255), (150, 200, 240), (200, 230, 255),
         (160, 210, 245), (170, 215, 250), (140, 195, 235)],            # light multicolor
    8:  [(0,   0,   0  ), (0,   80,  160), (180, 30,  30 ),
         (0,   40,  120), (140, 20,  20 ), (20,  60,  140)],            # black, blue, red
    9:  [(20,  20,  60 ), (60,  20,  80 ), (20,  60,  40 ),
         (40,  40,  80 ), (80,  20,  60 ), (20,  80,  60 )],            # dark multicolor
    10: [(180, 0,   0  ), (0,   150, 0  ), (0,   0,   200),
         (160, 0,   0  ), (0,   130, 0  ), (0,   0,   180)],            # RGB
    11: [(180, 30,  30 ), (30,  80,  180), (180, 30,  30 ),
         (30,  80,  180), (160, 20,  20 ), (20,  60,  160)],            # red, blue
    12: [(100, 100, 120), (120, 100, 110), (110, 120, 100),
         (90,  110, 120), (115, 105, 95 ), (105, 115, 110)],            # muted multicolor
    13: [(0,   80,  160), (180, 80,  0  ), (200, 30,  30 ),
         (0,   60,  140), (160, 60,  0  ), (180, 20,  20 )],            # blue, red, orange
    14: [(0,   60,  160), (30,  90,  180), (60,  120, 200),
         (10,  70,  170), (40,  100, 190), (20,  80,  175)],            # blue shades
    15: [(180, 30,  30 ), (220, 100, 30 ), (60,  160, 60 ),
         (160, 20,  20 ), (200, 80,  20 ), (40,  140, 40 )],            # red, orange, green
    16: [(255, 255, 255)] * 6,                                          # white (dark bg)
    17: [(30,  160, 80 ), (0,   100, 180), (60,  180, 100),
         (20,  140, 60 ), (0,   80,  160), (40,  160, 90 )],            # green, blue
    18: [(31,  119, 180), (255, 127, 14 ), (44,  160, 44 ),
         (214, 39,  40 ), (148, 103, 189), (23,  190, 207)],            # fallback bright
}

BACKGROUND_COLOR_BASES = {
    0: None,                  # transparent / white (use "white" string)
    1: (255, 255, 255),       # white
    2: (242, 242, 242),       # light gray
    3: (77,  77,  77),        # dark gray
    4: (237, 244, 255),       # light color
    5: (235, 235, 235),       # outer gray (unused in bar but kept for parity)
    6: (30,  30,  50),        # dark
}

# Named background lists kept for backward compat with background_color()
BACKGROUND_LIGHT_GRAY  = ["#f0f0f0", "#ebebebc0", "#e8e8e8", "#f2f2f2ed", "#edededf6"]
BACKGROUND_DARK_GRAY   = ["#8B8C8DC0", "#57676bf2", "#3d3d3de2", "#404040FF", "#2c2c2c"]
BACKGROUND_LIGHT_COLOR = ["#eef2ffe1", "#fff3e0cf", "#e8f5e9da", "#fce4ec", "#e3f2fddc", "#f3e5f5ca"]
BACKGROUND_DARK        = ["#4444817D", "#16213e", "#423a3a", "#00641E94", "#0a4e038f", "#083fb68f"]

### Validate sampling weights


In [169]:
def normalize_weights(weights: dict) -> dict:
    out = {}
    for param, codes in weights.items():
        total = sum(codes.values())
        out[param] = {k: v / total for k, v in codes.items()}
    return out


def sample_style(rng: np.random.Generator, weights: dict, overrides: dict | None = None) -> dict:
    style = {}
    for param, val_probs in weights.items():
        vals  = list(val_probs.keys())
        probs = np.array(list(val_probs.values()), dtype=float)
        probs /= probs.sum()
        style[param] = vals[int(rng.choice(len(vals), p=probs))]

    # Expand title_color group name → specific hex
    color_group = style.get("title_color")
    if color_group in TITLE_COLOR_PALETTES:
        palette = TITLE_COLOR_PALETTES[color_group]
        style["title_color"] = palette[int(rng.integers(0, len(palette)))]

        

    if overrides:
        style.update(overrides)
    return harmonize_style(style)

In [170]:
OBSERVED_WEIGHTS = normalize_weights(SAMPLING_WEIGHTS)

weight_check = pd.DataFrame([
    {
        "parameter": param,
        "n_codes": len(codes),
        "raw_sum": round(sum(codes.values()), 10),
        "normalized_sum": round(sum(OBSERVED_WEIGHTS[param].values()), 10),
    }
    for param, codes in SAMPLING_WEIGHTS.items()
])

weight_check

,parameter,n_codes,raw_sum,normalized_sum
0,is_vertical,2,1.0,1.0
1,title_present,2,1.0,1.0
2,title_location,3,1.0,1.0
3,title_color,2,1.0,1.0
4,title_size,2,1.0,1.0
5,title_size_large,2,1.0,1.0
6,subtitle_present,2,1.0,1.0
7,outline_chart,2,1.0,1.0
8,legend_location,6,1.0,1.0
9,legend_title,2,1.0,1.0


## 2. Data sampling


In [171]:
def sample_bar_data_from_df(
    df: pd.DataFrame,
    spec: dict,
    rng: np.random.Generator,
    style: dict,
    min_total: float = 1.0,
) -> tuple[pd.DataFrame, dict] | None:
    numeric_cols = [c for c in spec["numeric_cols"] if c in df.columns]
    group_cols   = [c for c in spec["group_cols"]   if c in df.columns]
    date_col     = spec.get("date_col")

    sample_bin = style.get("sample_bin", 8)
    n_bars = int(sample_bin) if isinstance(sample_bin, int) and sample_bin >= 2 else 8
    n_bars = max(2, min(n_bars, 30))

    # ── London borough/sector dataset ────────────────────────────────────────
    if {"BOROUGH", "SECTOR", "YEAR", "EMPLOYEE_JOBS"}.issubset(df.columns):
        strategy = str(rng.choice(
            ["sector_totals", "borough_totals", "sector_year", "borough_year"],
            p=[0.35, 0.35, 0.15, 0.15]
        ))

        if strategy == "sector_totals":
            # Total employee jobs per sector across all years
            series = df.groupby("SECTOR")["EMPLOYEE_JOBS"].sum()
            series = series[series > 0].sort_values(ascending=False).head(n_bars)
            if len(series) < 2:
                return None
            plot_df = series.reset_index()
            plot_df.columns = ["SECTOR", "EMPLOYEE_JOBS"]
            context = {
                "value_col": "EMPLOYEE_JOBS",
                "category_col": "SECTOR",
                "strategy": strategy,
                "n_bars": len(plot_df),
                "total": float(series.sum()),
            }

        elif strategy == "borough_totals":
            # Total employee jobs per borough across all years
            series = df.groupby("BOROUGH")["EMPLOYEE_JOBS"].sum()
            series = series[series > 0].sort_values(ascending=False).head(n_bars)
            if len(series) < 2:
                return None
            plot_df = series.reset_index()
            plot_df.columns = ["BOROUGH", "EMPLOYEE_JOBS"]
            context = {
                "value_col": "EMPLOYEE_JOBS",
                "category_col": "BOROUGH",
                "strategy": strategy,
                "n_bars": len(plot_df),
                "total": float(series.sum()),
            }

        elif strategy == "sector_year":
            # Jobs per sector for a randomly chosen year
            year_candidates = sorted(df["YEAR"].dropna().unique().tolist())
            if not year_candidates:
                return None
            year_val = int(rng.choice(year_candidates))
            sub = df[df["YEAR"] == year_val]
            series = sub.groupby("SECTOR")["EMPLOYEE_JOBS"].sum()
            series = series[series > 0].sort_values(ascending=False).head(n_bars)
            if len(series) < 2:
                return None
            plot_df = series.reset_index()
            plot_df.columns = ["SECTOR", "EMPLOYEE_JOBS"]
            context = {
                "value_col": "EMPLOYEE_JOBS",
                "category_col": "SECTOR",
                "strategy": strategy,
                "year": year_val,
                "n_bars": len(plot_df),
                "total": float(series.sum()),
            }

        else:  # borough_year
            # Jobs per borough for a randomly chosen year
            year_candidates = sorted(df["YEAR"].dropna().unique().tolist())
            if not year_candidates:
                return None
            year_val = int(rng.choice(year_candidates))
            sub = df[df["YEAR"] == year_val]
            series = sub.groupby("BOROUGH")["EMPLOYEE_JOBS"].sum()
            series = series[series > 0].sort_values(ascending=False).head(n_bars)
            if len(series) < 2:
                return None
            plot_df = series.reset_index()
            plot_df.columns = ["BOROUGH", "EMPLOYEE_JOBS"]
            context = {
                "value_col": "EMPLOYEE_JOBS",
                "category_col": "BOROUGH",
                "strategy": strategy,
                "year": year_val,
                "n_bars": len(plot_df),
                "total": float(series.sum()),
            }

        return plot_df, context

    # ── General datasets ──────────────────────────────────────────────────────
    if not numeric_cols or not group_cols:
        return None

    value_col    = str(rng.choice(numeric_cols))
    category_col = str(rng.choice(group_cols))

    strategy = str(rng.choice(["group", "time_window", "filtered_group"],
                               p=[0.40, 0.35, 0.25]))
    context = {"value_col": value_col, "strategy": strategy}

    if strategy == "time_window" and date_col and date_col in df.columns:
        window_months = int(rng.choice([1, 3, 6, 12], p=[0.30, 0.35, 0.20, 0.15]))
        monthly_totals = df.groupby(date_col)[value_col].sum()
        valid_months   = monthly_totals[monthly_totals > min_total].index.to_numpy()
        if len(valid_months) == 0:
            return None
        start = pd.to_datetime(rng.choice(valid_months))
        end   = start + pd.DateOffset(months=window_months)
        dfw   = df[(df[date_col] >= start) & (df[date_col] < end)].copy()
        if len(dfw) == 0:
            return None
        series = dfw.groupby(category_col)[value_col].sum()
        context.update({"category_col": category_col, "start": start,
                        "end": end, "window_months": window_months})

    elif strategy == "filtered_group" and len(group_cols) >= 2:
        other_cols = [c for c in group_cols if c != category_col]
        filter_col = str(rng.choice(other_cols))
        candidates = np.array(df[filter_col].dropna().unique(), dtype=str)
        rng.shuffle(candidates)
        dfw = None
        for val in candidates[:10]:
            tmp        = df[df[filter_col] == val].copy()
            tmp_series = tmp.groupby(category_col)[value_col].sum()
            tmp_series = tmp_series[tmp_series > 0]
            if len(tmp_series) >= 2 and tmp_series.sum() > min_total:
                dfw = tmp
                filter_val = str(val)
                break
        if dfw is None:
            return None
        series = dfw.groupby(category_col)[value_col].sum()
        context.update({"category_col": category_col, f"filter_{filter_col}": filter_val})

    else:  # "group"
        series = df.groupby(category_col)[value_col].sum()
        context["category_col"] = category_col

    series = series[series > 0].sort_values(ascending=False)
    if len(series) < 2:
        return None
    series = series.head(n_bars)
    if len(series) < 2:
        return None

    total = float(series.sum())
    if total <= min_total:
        return None

    plot_df = series.reset_index()
    plot_df.columns = [category_col, value_col]
    context.update({
        "category_col": category_col,
        "n_bars": int(len(plot_df)),
        "total":  total,
    })
    return plot_df, context

## 3. Style helpers


In [172]:
# ── WCAG contrast-aware helper functions ─────────────────────────────────────
# Ported from line generator for consistent colour sampling across chart types.

# ── Helper functions ──────────────────────────────────────────────────────────

def _hex_to_rgb(hex_color: str) -> tuple[int, int, int]:
    hex_color = hex_color.lstrip("#")
    return int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)


def _relative_luminance(hex_color: str) -> float:
    """WCAG relative luminance of a hex color, in [0, 1]."""
    r, g, b = (_hex_to_rgb(hex_color))
    def linearize(c):
        c /= 255
        return c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4
    return 0.2126 * linearize(r) + 0.7152 * linearize(g) + 0.0722 * linearize(b)


def _contrast_ratio(hex_a: str, hex_b: str) -> float:
    """WCAG contrast ratio between two hex colors."""
    la = _relative_luminance(hex_a)
    lb = _relative_luminance(hex_b)
    lighter, darker = max(la, lb), min(la, lb)
    return (lighter + 0.05) / (darker + 0.05)

def _sample_with_contrast(base: tuple[int, int, int], bg: str,
                           rng: np.random.Generator, v: int = 30) -> str:
    check_contrast = bg not in {"none", "transparent"}
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate
    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def get_background_color(style: dict, rng: np.random.Generator | None = None) -> str:
    """Background color — handles both bar's string keys and numeric codes."""
    rng  = rng or np.random.default_rng()
    code = style.get("background", "white")
    # String-keyed (bar generator style)
    if code == "white":
        return "#ffffff"
    if code == "light_gray":
        return BACKGROUND_LIGHT_GRAY[int(rng.integers(0, len(BACKGROUND_LIGHT_GRAY)))]
    if code == "dark_gray":
        return BACKGROUND_DARK_GRAY[int(rng.integers(0, len(BACKGROUND_DARK_GRAY)))]
    if code == "light_color":
        return BACKGROUND_LIGHT_COLOR[int(rng.integers(0, len(BACKGROUND_LIGHT_COLOR)))]
    if code == "dark":
        return BACKGROUND_DARK[int(rng.integers(0, len(BACKGROUND_DARK)))]
    # Numeric-keyed fallback
    try:
        code_int = int(code)
        base = BACKGROUND_COLOR_BASES.get(code_int)
        if base is None:
            return "#ffffff"
        v = 8
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        return f"#{r:02x}{g:02x}{b:02x}"
    except (ValueError, TypeError):
        return "#ffffff"


def get_foreground_color(style: dict) -> str:
    bg = style.get("background", "white")
    return "white" if bg in {"dark_gray", "dark", 3, 6} else "black"


def get_title_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("title_color", 0))
    base = TITLE_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    # Skip contrast check for transparent background
    check_contrast = bg not in {"none", "transparent"}

    v = 30
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    # Fallback: return pure black or white depending on background luminance
    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def get_axis_color(style: dict, rng: np.random.Generator | None = None) -> str | None:
    """Returns the axis line color, or None for code 3 (no axes)."""
    code = int(style.get("axis_color", 0))
    if code == 3:
        return None
    base = AXIS_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)
    return _sample_with_contrast(base, bg, rng, v=20)

def get_legend_title_fontsize(style: dict, rng: np.random.Generator | None = None) -> int | None:
    if int(style.get("legend_title_size", 0)) != 1:
        return None
    rng = rng or np.random.default_rng()
    return int(np.clip(11 + rng.integers(-1, 2), 8, 16))


def get_title_fontsize(style: dict, rng: np.random.Generator | None = None) -> int:
    code = int(style.get("title_size", 0))
    base = TITLE_SIZE_BASES.get(code, 16)
    rng  = rng or np.random.default_rng()
    return int(np.clip(base + rng.integers(-2, 3), 8, 32))


def get_title_location(style: dict):
    """Returns (label, x_pos, halign, anchor)."""
    return TITLE_LOCATIONS.get(int(style.get("title_location", 1)),
                               ("center", 0.50, "center", "middle"))


def get_axis_text_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("axis_text_color", 0))
    if code == 1:
        return get_title_color(style, rng=rng)
    if code == 3:
        bg = get_background_color(style, rng=rng)
        return bg if bg not in {"none", "transparent"} else "white"
    return AXIS_TEXT_COLORS.get(code, "black") or "black"


def get_gridline_style(style: dict, rng: np.random.Generator | None = None) -> tuple[str, str]:
    """Returns (color, linestyle) where linestyle is 'solid' or 'dashed'."""
    code = int(style.get("gridline_color", 1))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    base = GRIDLINE_COLOR_BASES.get(code, (140, 140, 140))
    linestyle = "dashed" if code == 5 else "solid"

    color = _sample_with_contrast(base, bg, rng, v=15)
    return color, linestyle


def get_label_color(style: dict, rng: np.random.Generator | None = None,
                    line_color: str | None = None) -> str:
    code = int(style.get("label_color", 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    if code == 1:
        return get_title_color(style, rng=rng)

    if code == 2:
        return _sample_with_contrast((0, 0, 0), bg, rng, v=20)

    if code == 3:
        return line_color or BRIGHT_COLORS[0]

    if code == 4:
        base = 140
        v    = 40
        gray = int(np.clip(base + rng.integers(-v, v + 1), 80, 220))
        hex_gray = f"#{gray:02x}{gray:02x}{gray:02x}"
        # Retry if contrast is poor
        bg_check = bg not in {"none", "transparent"}
        for _ in range(20):
            gray = int(np.clip(base + rng.integers(-v, v + 1), 80, 220))
            hex_gray = f"#{gray:02x}{gray:02x}{gray:02x}"
            if not bg_check or _contrast_ratio(hex_gray, bg) >= 3.0:
                return hex_gray
        return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"

    # code 0 or fallback — black with contrast check
    return _sample_with_contrast((0, 0, 0), bg, rng, v=10)

def get_legend_fill_and_outline(style: dict, rng: np.random.Generator | None = None) -> tuple[str | None, bool]:
    """Returns (fill_color, has_outline). Encodes all legend_fill codes in one place."""
    rng  = rng or np.random.default_rng()
    code = int(style.get("legend_fill", 0))

    if code == 0:
        return None, False

    if code == 1:
        # Gray fill, no outline — sample a gray shade for variation
        base = 180
        v    = 30
        gray = int(np.clip(base + rng.integers(-v, v + 1), 120, 235))
        return f"#{gray:02x}{gray:02x}{gray:02x}", False

    if code == 2:
        # No fill, no outline
        return None, False

    if code in {3, 4}:
        # White fill with black outline
        return "#ffffff", True

    return None, False


def get_legend_text_color(style: dict, rng: np.random.Generator | None = None) -> str | None:
    code = int(style.get("legend_text_color", 0))
    if code == 1:
        return get_title_color(style, rng=rng)
    if code == 2:
        return None

    base_color = {0: "#000000", 3: "#555555", 4: "#ffffff", 5: "#888888", 6: "#333333"}.get(code, "#000000")

    bg = get_background_color(style)
    if bg in {"none", "transparent"}:
        return base_color

    r, g, b = _hex_to_rgb(base_color)
    rng = rng or np.random.default_rng()
    v = 20
    for _ in range(20):
        rc = int(np.clip(r + rng.integers(-v, v + 1), 0, 255))
        gc = int(np.clip(g + rng.integers(-v, v + 1), 0, 255))
        bc = int(np.clip(b + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{rc:02x}{gc:02x}{bc:02x}"
        if _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"

LEGEND_TITLE_COLOR_BASES = {
    0: (0, 0, 0),        # black
    1: (100, 100, 100),  # gray
    2: (255, 255, 255),  # white
}

def get_legend_title_color(style: dict, rng: np.random.Generator | None = None) -> str:
    code = int(style.get("legend_title_color", 0))
    base = LEGEND_TITLE_COLOR_BASES.get(code, (0, 0, 0))
    rng  = rng or np.random.default_rng()
    bg   = get_background_color(style)

    check_contrast = bg not in {"none", "transparent"}

    v = 20
    for _ in range(20):
        r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
        g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
        b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
        candidate = f"#{r:02x}{g:02x}{b:02x}"
        if not check_contrast or _contrast_ratio(candidate, bg) >= 3.0:
            return candidate

    return "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"


def color_list(style: dict, n: int, rng: np.random.Generator | None = None) -> list:
    code  = int(style.get("palette_type", 2))
    rng   = rng or np.random.default_rng()
    bases = PALETTE_BASES.get(code, PALETTE_BASES[2])
    bg    = get_background_color(style)
    v     = 15

    def _too_similar(candidate: str, existing: list[str], threshold: int = 60) -> bool:
        cr, cg, cb = _hex_to_rgb(candidate)
        for ex in existing:
            er, eg, eb = _hex_to_rgb(ex)
            if abs(cr - er) + abs(cg - eg) + abs(cb - eb) < threshold:
                return True
        return False

    colors = []
    for i in range(n):
        base = bases[i % len(bases)]
        chosen = None
        for _ in range(40):
            r = int(np.clip(base[0] + rng.integers(-v, v + 1), 0, 255))
            g = int(np.clip(base[1] + rng.integers(-v, v + 1), 0, 255))
            b = int(np.clip(base[2] + rng.integers(-v, v + 1), 0, 255))
            candidate = f"#{r:02x}{g:02x}{b:02x}"
            bg_ok      = bg in {"none", "transparent"} or _contrast_ratio(candidate, bg) >= 2.0
            similar_ok = not _too_similar(candidate, colors)
            if bg_ok and similar_ok:
                chosen = candidate
                break
        if chosen is None:
            # Fallback — spread evenly through hue space
            chosen = "#ffffff" if _relative_luminance(bg) < 0.5 else "#000000"
        colors.append(chosen)

    return colors


def line_pattern_for(style: dict, idx: int, renderer: str = "mpl") -> str:
    code = int(style.get("line_pattern", 0))
    if code == 3:   # mixed
        sub = idx % 3
        return LINE_PATTERNS_MPL.get(sub, "-") if renderer == "mpl" else LINE_PATTERNS_PLY.get(sub, "solid")
    return LINE_PATTERNS_MPL.get(code, "-") if renderer == "mpl" else LINE_PATTERNS_PLY.get(code, "solid")


def marker_for(style: dict, idx: int):
    mode = POINT_SHAPE_MODES.get(int(style.get("point_shape_mode", 0)), "dots")
    if mode == "none":
        return None
    if mode == "by_line":
        return ["o", "s", "^", "D", "P", "X"][idx % 6]
    return MPL_MARKERS.get(mode, "o")


def point_color(line_color: str, style: dict, idx: int) -> str:
    mode = POINT_SAME_COLORS.get(int(style.get("point_same_color", 1)), "same")
    if mode == "none":
        return line_color
    if mode == "different":
        return BRIGHT_COLORS[(idx + 2) % len(BRIGHT_COLORS)]
    if mode == "shade":
        return "#999999"
    return line_color


def legend_location_mpl(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    if not entry:
        return None, None
    _, loc, bbox, _ = entry
    return loc, bbox

def legend_orient_altair(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[0] if entry else None

def legend_coords_plotly(style: dict) -> dict:
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[3] if entry else {}

def legend_orient_altair(style: dict):
    entry = LEGEND_ORIENTATIONS.get(int(style.get("legend_orientation", 2)))
    return entry[0] if entry else None


def apply_gridlines_mpl(ax, style: dict, rng: np.random.Generator | None = None):
    grid = int(style.get("gridlines", 0))
    if grid == 0:
        ax.grid(False)
        return

    color, linestyle = get_gridline_style(style, rng=rng)
    mpl_linestyle = "--" if linestyle == "dashed" else "-"

    if grid == 2:
        ax.yaxis.grid(True, color=color, linewidth=0.6, linestyle=mpl_linestyle, alpha=0.75, zorder=0)
        ax.xaxis.grid(False)
    elif grid in {1, 3, 4}:
        width = 0.4 if grid == 3 else 0.8 if grid == 4 else 0.6
        ax.grid(True, color=color, linewidth=width, linestyle=mpl_linestyle, alpha=0.75, zorder=0)
    else:
        ax.grid(False)


def apply_outline_mpl(ax, fig, style: dict, rng: np.random.Generator | None = None):
    outline  = int(style.get("chart_outline", 1))
    fg       = get_foreground_color(style)
    axis_c   = get_axis_color(style, rng=rng)

    if axis_c is None:
        # code 3 — no axes at all
        for s in ax.spines.values():
            s.set_visible(False)
        ax.tick_params(left=False, bottom=False)
        if int(style.get("image_outline", 0)) == 1:
            fig.patch.set_edgecolor(fg)
            fig.patch.set_linewidth(1.2)
        return

    for spine in ax.spines.values():
        spine.set_color(axis_c)

    if outline == 0:
        for s in ax.spines.values():
            s.set_visible(False)
    elif outline == 1:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
    elif outline == 2:
        pass  # full box
    elif outline == 4:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(False)
        ax.yaxis.set_visible(False)

    if int(style.get("image_outline", 0)) == 1:
        fig.patch.set_edgecolor(fg)
        fig.patch.set_linewidth(1.2)

def color_list_bar(style: dict, n: int, rng: np.random.Generator | None = None) -> list[str]:
    """
    Bar-chart wrapper for color_list.
    style["palette_type"] is an int code mapping to PALETTE_BASES.
    """
    return color_list(style, max(1, n), rng=rng)


def wrap_labels(labels: list[str], max_chars: int = 10) -> list[str]:
    """Wrap long labels by inserting newlines at word boundaries."""
    import textwrap
    return ["\n".join(textwrap.wrap(str(l), width=max_chars)) for l in labels]


def needs_rotation(labels: list[str], fig_width_inches: float, fontsize: int = 9) -> bool:
    avg_char_width = fontsize * 0.6 / 72  # inches per character
    n = max(len(labels), 1)
    max_label_width = max(len(str(l)) for l in labels) * avg_char_width
    bar_width = fig_width_inches / n
    return max_label_width > bar_width

In [173]:
COLOR_PALETTE_HEX = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2",
    "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#637939", "#8c6d31", "#843c39", "black",
]

# Altair-valid scheme names only
LIGHT_SCHEMES = [
    "tableau10", "tableau20", "category10", "category20", "category20b", "category20c",
    "set1", "set2", "set3", "dark2", "paired", "accent", "observable10",
]

DARK_SCHEMES = [
    "pastel1", "pastel2", "set3", "tableau20", "category20",
    "category20b", "category20c", "paired",
]

SCHEME_NAMES = LIGHT_SCHEMES  # kept for backward compat

MPL_SCHEMES = {
    "tableau10":    "tab10",
    "tableau20":    "tab20",
    "category10":   "tab10",
    "category20":   "tab20",
    "category20b":  "tab20b",
    "category20c":  "tab20c",
    "set1":         "Set1",
    "set2":         "Set2",
    "set3":         "Set3",
    "dark2":        "Dark2",
    "paired":       "Paired",
    "accent":       "Accent",
    "observable10": "tab10",
    "pastel1":      "Pastel1",
    "pastel2":      "Pastel2",
}

_ALTAIR_VALID_ORIENTS = {
    "none", "left", "right", "top", "bottom",
    "top-left", "top-right", "bottom-left", "bottom-right",
}


def pick_scheme(style: dict, rng: np.random.Generator) -> str:
    if style.get("background") in {"dark_gray", "dark"}:
        return DARK_SCHEMES[int(rng.integers(0, len(DARK_SCHEMES)))]
    return LIGHT_SCHEMES[int(rng.integers(0, len(LIGHT_SCHEMES)))]


def new_chart_id(prefix="bar"):
    stamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    return f"{prefix}_{stamp}_{uuid4().hex[:8]}"


def safe_slug(value):
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_") or "value"

DARK_BACKGROUNDS = {"dark_gray", "dark"}

def harmonize_style(style: dict) -> dict:
    style = dict(style)
    if not style.get("title_present", True):
        style["title_location"] = "none"
        style["subtitle_present"] = False
    if style.get("title_size_large"):
        style["title_size"] = "large"
    if style.get("legend_location") == "no legend":
        style["legend_title"]       = False
        style["legend_title_color"] = False
        style["legend_outline"]     = False
    if style.get("gridlines_both"):
        style["gridlines"] = True
    if style.get("x_labels_vertical"):
        style["x_labels_orientation"] = "vertical"
    elif style.get("x_labels_diagonal"):
        style["x_labels_orientation"] = "diagonal"
    if style.get("y_labels_vertical"):
        style["y_labels_orientation"] = "vertical"
    elif style.get("y_labels_diagonal"):
        style["y_labels_orientation"] = "diagonal"
    if style.get("background") in DARK_BACKGROUNDS:
        style["title_color"]    = "#ffffff"
        style["labels_colored"] = False
        style["_force_white_text"] = True
    else:
        style.pop("_force_white_text", None)
    return style


def figure_size(style):
    return {"square": (6, 6), "horizontal": (8, 5), "vertical": (5, 8)}.get(
        style.get("aspect_ratio", "horizontal"), (8, 5))


def axis_angle(orientation, backend="matplotlib"):
    if orientation in {"diagonal"}:
        return -45 if backend == "altair" else 45
    if orientation in {"vertical"}:
        return -90 if backend == "altair" else 90
    return 0


def labels_visible(orientation):
    return orientation != "hidden"


def label_color(style, rng: np.random.Generator):
    if style.get("_force_white_text"):
        return "white"
    if style.get("labels_colored"):
        bg = background_color(style)
        # Random vivid base color — spread across hue space
        import colorsys
        hue = float(rng.uniform(0, 1))
        sat = float(rng.uniform(0.5, 0.9))
        val = float(rng.uniform(0.3, 0.9))
        r, g, b = colorsys.hsv_to_rgb(hue, sat, val)
        base = (int(r * 255), int(g * 255), int(b * 255))
        return _sample_with_contrast(base, bg, rng, v=20)
    return "black"


def background_color(style, rng: np.random.Generator | None = None):
    if "_bg_color" in style:
        return style["_bg_color"]
    return get_background_color(style, rng=rng)

def title_font_size(style, rng: np.random.Generator):
    if style.get("title_size") == "large":
        return int(rng.integers(20, 25))
    if style.get("title_size") == "small":
        return int(rng.integers(10, 13))
    return int(rng.integers(12, 19))


def title_location_mpl(style):
    return {"center": "center", "left": "left", "right": "right"}.get(
        style.get("title_location", "center"), "center")


def title_anchor_altair(style):
    return {"center": "middle", "left": "start", "right": "end"}.get(
        style.get("title_location", "center"), "middle")


def legend_location_mpl(location):
    return {
        "left":      "center left",  "right":     "center right",
        "top":       "upper center", "bottom":    "lower center",
        "top-left":  "upper left",   "top-right": "upper right",
        "no legend": None,
    }.get(location, "best")


def legend_location_altair(location):
    if location in {"no legend", None}:
        return None
    return location if location in _ALTAIR_VALID_ORIENTS else "right"


def grid_axes(style):
    if not style.get("gridlines"):
        return False, False
    if style.get("gridlines_both"):
        return True, True
    return (False, True) if style.get("is_vertical") else (True, False)


def bar_colors(style: dict, n: int, rng: np.random.Generator) -> list[str]:
    if not style.get("colored_bars", True):
        return ["black"] * n

    scheme    = pick_scheme(style, rng)
    mpl_name  = MPL_SCHEMES.get(scheme, "tab10")
    cmap      = plt.get_cmap(mpl_name)
    return [mcolors.to_hex(cmap(i / max(len(range(n)) - 1, 1))) for i in range(n)]

## 4. Title generation


In [174]:
# UNCOMMENT FOR GROQ
# _groq_client = Groq()  # reads GROQ_API_KEY from env

MAX_TITLE_CHARS = 60


def make_title(context: dict, style: dict | None = None) -> tuple[str, str | None]:
    value_col    = context.get("value_col", "Value")
    category_col = context.get("category_col", "Category")

    filter_parts = [f"{k[7:]}: {v}" for k, v in context.items() if k.startswith("filter_")]
    filter_desc  = f" (filtered to {', '.join(filter_parts)})" if filter_parts else ""

    time_desc = ""
    if "start" in context:
        start_str = pd.to_datetime(context["start"]).strftime("%B %Y")
        end_str   = pd.to_datetime(context["end"]).strftime("%B %Y")
        time_desc = f", covering {start_str} to {end_str}"

    need_subtitle = style is not None and style.get("subtitle_present", False)

    system_prompt = (
        "You generate short, realistic chart titles for bar charts — the kind you'd see "
        "in a business report or dashboard. Keep titles under 60 characters. "
        "Be concise and natural. No quotes, no markdown."
    )
    user_prompt = (
        f"Generate a chart title for a bar chart showing {value_col} "
        f"by {category_col}{filter_desc}{time_desc}.\n"
    )
    if need_subtitle:
        user_prompt += (
            "Also generate a short subtitle (one line, adds context or time range detail). "
            "Respond in this exact format:\nTITLE: <title here>\nSUBTITLE: <subtitle here>"
        )
    else:
        user_prompt += "Respond with just the title, nothing else."

    try:
        response = ollama.chat(
            model="llama3.2",  # or whichever model you have pulled
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
        )
        raw = response["message"]["content"].strip()

        if need_subtitle and "TITLE:" in raw:
            lines = {k.strip(): v.strip() for k, v in
                     (line.split(":", 1) for line in raw.splitlines() if ":" in line)}
            return lines.get("TITLE", raw)[:MAX_TITLE_CHARS], lines.get("SUBTITLE", "")[:MAX_TITLE_CHARS]

        return raw[:MAX_TITLE_CHARS], None

    except Exception:
        fallback_title = f"{value_col} by {category_col}"[:MAX_TITLE_CHARS]
        fallback_sub   = "Business report" if need_subtitle else None
        return fallback_title, fallback_sub


## 5. Shared plotting helpers


In [175]:
def apply_mpl_common_style(fig, ax, style, title, subtitle, rng):
    bg = background_color(style, rng)
    ax.set_facecolor(bg)
    fig.patch.set_facecolor(bg)

    if style.get("title_present") and title:
        title_text = f"{title}\n{subtitle}" if subtitle else title
        ax.set_title(
            title_text,
            loc=title_location_mpl(style),
            color=style.get("title_color", "#000000"),
            fontsize=title_font_size(style, rng),
        )

    if style.get("outline_chart"):
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color("black")
            spine.set_linewidth(1)
    else:
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_visible(True)
        ax.spines["bottom"].set_visible(True)

    grid_x, grid_y = grid_axes(style)
    if grid_x or grid_y:
        axis = "both" if (grid_x and grid_y) else ("x" if grid_x else "y")
        ax.grid(True, axis=axis, linestyle="-", alpha=0.3, color="gray")
        ax.set_axisbelow(True)
    else:
        ax.grid(False)

    lx      = label_color(style, rng)
    x_angle = axis_angle(style.get("x_labels_orientation", "horizontal"))
    y_angle = axis_angle(style.get("y_labels_orientation", "horizontal"))

    if labels_visible(style.get("x_labels_orientation", "horizontal")):
        plt.setp(ax.get_xticklabels(), rotation=x_angle, color=lx)
        ax.tick_params(axis="x", colors=lx,
                       length=3 if style.get("ticks") in ["x", "both"] else 0)
    else:
        ax.set_xticklabels([])
        ax.tick_params(axis="x", length=0)

    if labels_visible(style.get("y_labels_orientation", "horizontal")):
        plt.setp(ax.get_yticklabels(), rotation=y_angle, color=lx)
        ax.tick_params(axis="y", colors=lx,
                       length=3 if style.get("ticks") in ["y", "both"] else 0)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", length=0)


def add_mpl_bar_labels(ax, plot_df, value_col, style):
    mode = style.get("direct_labeling", "none")
    if mode == "none":
        return
    if mode == "direct_no_axis":
        if style.get("is_vertical"):
            ax.set_yticklabels([])
        else:
            ax.set_xticklabels([])
    for patch, value in zip(ax.patches, plot_df[value_col]):
        label = str(int(round(value))) if abs(value - round(value)) < 1e-9 else f"{value:,.1f}"
        if style.get("is_vertical"):
            x = patch.get_x() + patch.get_width() / 2
            if style.get("direct_label_position") == "center":
                y, va = patch.get_height() / 2, "center"
            elif style.get("direct_label_position") == "outside":
                y, va = patch.get_height(), "bottom"
            else:
                y, va = max(1, patch.get_height() * 0.05), "bottom"
            ax.text(x, y, label, ha="center", va=va, fontsize=9)
        else:
            y = patch.get_y() + patch.get_height() / 2
            if style.get("direct_label_position") == "center":
                x, ha = patch.get_width() / 2, "center"
            elif style.get("direct_label_position") == "outside":
                x, ha = patch.get_width(), "left"
            else:
                x, ha = max(1, patch.get_width() * 0.05), "left"
            ax.text(x, y, label, va="center", ha=ha, fontsize=9)


def add_mpl_legend(ax, plot_df, category_col, colors, style):
    loc = legend_location_mpl(style.get("legend_location", "no legend"))
    if loc is None or not style.get("colored_bars", True):
        existing = ax.get_legend()
        if existing:
            existing.remove()
        return
    handles = [mpatches.Patch(color=c, label=l)
               for c, l in zip(colors, plot_df[category_col])]
    kwargs = {"loc": loc, "title": category_col if style.get("legend_title") else None}
    if style.get("legend_outline"):
        kwargs.update({"frameon": True, "edgecolor": "black", "framealpha": 1})
    else:
        kwargs.update({"frameon": False})
    legend = ax.legend(handles=handles, **kwargs)
    if style.get("legend_title_color") and legend.get_title():
        legend.get_title().set_color("#555555")


## 6. Bar renderers


In [176]:
def render_bar_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng):
    fig, ax  = plt.subplots(figsize=figure_size(style))
    if style.get("colored_bars", True):
        #scheme = SCHEME_NAMES[int(rng.integers(0, len(SCHEME_NAMES)))]
        scheme = pick_scheme(style, rng)
        cmap   = plt.get_cmap(MPL_SCHEMES[scheme])
        colors = [mcolors.to_hex(cmap(i / max(len(plot_df) - 1, 1))) for i in range(len(plot_df))]
    else:
        colors = ["black"] * len(plot_df)
    edgecolor = "black" if style.get("outline_bars") else "none"
    linewidth =  1      if style.get("outline_bars") else 0
    if style.get("is_vertical"):
        bars = ax.bar(plot_df[category_col], plot_df[value_col],
                      color=colors, edgecolor=edgecolor, linewidth=linewidth)
    else:
        bars = ax.barh(plot_df[category_col], plot_df[value_col],
                       color=colors, edgecolor=edgecolor, linewidth=linewidth)
    if style.get("rounded_corners"):
        for patch in bars:
            patch.set_alpha(0.92)
    ax.set_xlabel("")
    ax.set_ylabel("")
    apply_mpl_common_style(fig, ax, style, title, subtitle, rng)
    add_mpl_bar_labels(ax, plot_df, value_col, style)
    add_mpl_legend(ax, plot_df, category_col, colors, style)

    if style.get("is_vertical"):
        labels = [t.get_text() for t in ax.get_xticklabels()]
        if labels and needs_rotation(labels, figure_size(style)[0]):
            wrapped = wrap_labels(labels)
            ax.set_xticks(ax.get_xticks())
            ax.set_xticklabels(wrapped, ha="center")

    fig.tight_layout()
    return fig


def render_bar_seaborn(plot_df, category_col, value_col, title, subtitle, style, rng):
    if sns is None:
        raise ImportError("seaborn is not installed")
    fig, ax   = plt.subplots(figsize=figure_size(style))
    edgecolor = "black" if style.get("outline_bars") else "none"
    linewidth =  1      if style.get("outline_bars") else 0
    plot_args = {"edgecolor": edgecolor, "linewidth": linewidth}
    if style.get("colored_bars", True):
        #scheme = SCHEME_NAMES[int(rng.integers(0, len(SCHEME_NAMES)))]
        scheme = pick_scheme(style, rng)
        plot_args.update({"hue": category_col, "palette": MPL_SCHEMES[scheme], "dodge": False})
    else:
        plot_args.update({"color": "black"})
    if style.get("is_vertical"):
        sns.barplot(data=plot_df, x=category_col, y=value_col, ax=ax, **plot_args)
    else:
        sns.barplot(data=plot_df, x=value_col, y=category_col, orient="h", ax=ax, **plot_args)
    colors = [p.get_facecolor() for p in ax.patches]
    ax.set_xlabel("")
    ax.set_ylabel("")
    apply_mpl_common_style(fig, ax, style, title, subtitle, rng)
    add_mpl_bar_labels(ax, plot_df, value_col, style)
    if style.get("colored_bars", True):
        existing = ax.get_legend()
        if existing:
            existing.remove()
        add_mpl_legend(ax, plot_df, category_col, colors, style)

    if style.get("is_vertical"):
        labels = [t.get_text() for t in ax.get_xticklabels()]
        if labels and needs_rotation(labels, figure_size(style)[0]):
            wrapped = wrap_labels(labels)
            ax.set_xticks(ax.get_xticks())
            ax.set_xticklabels(wrapped, ha="center")

    fig.tight_layout()
    return fig


def legend_location_plotly(location):
    return {
        "left":      {"x": 0.01, "y": 0.5,   "xanchor": "left",   "yanchor": "middle", "orientation": "v"},
        "right":     {"x": 0.99, "y": 0.5,   "xanchor": "right",  "yanchor": "middle", "orientation": "v"},
        "top":       {"x": 0.5,  "y": 1.02,  "xanchor": "center", "yanchor": "bottom", "orientation": "h"},
        "bottom":    {"x": 0.5,  "y": -0.15, "xanchor": "center", "yanchor": "top",    "orientation": "h"},
        "top-left":  {"x": 0.01, "y": 1.02,  "xanchor": "left",   "yanchor": "bottom", "orientation": "h"},
        "top-right": {"x": 0.99, "y": 1.02,  "xanchor": "right",  "yanchor": "bottom", "orientation": "h"},
    }.get(location, {"x": 1.0, "y": 1.0, "xanchor": "right", "yanchor": "top", "orientation": "v"})


def color_to_plotly(color):
    try:
        return mcolors.to_hex(color)
    except Exception:
        return str(color)


def render_bar_plotly(plot_df, category_col, value_col, title, subtitle, style, rng):
    global go
    if go is None:
        go = ensure_module("plotly.graph_objects", "plotly")

    fig        = go.Figure()
    if style.get("colored_bars", True):
        scheme     = SCHEME_NAMES[int(rng.integers(0, len(SCHEME_NAMES)))]
        cmap       = plt.get_cmap(MPL_SCHEMES[scheme])
        colors     = [mcolors.to_hex(cmap(i / max(len(plot_df) - 1, 1))) for i in range(len(plot_df))]
    else:
        colors = ["black"] * len(plot_df)
    showlegend = style.get("legend_location") != "no legend" and style.get("colored_bars", True)
    text_mode  = style.get("direct_labeling", "none") != "none"
    marker_line_color = "black" if style.get("outline_bars") else "rgba(0,0,0,0)"
    marker_line_width = 1       if style.get("outline_bars") else 0
    text_pos = {"base": "inside", "center": "inside", "outside": "outside"}.get(
        style.get("direct_label_position", "base"), "inside")

    for idx, row in plot_df.reset_index(drop=True).iterrows():
        color     = colors[idx % len(colors)]
        val_label = (str(int(round(row[value_col])))
                     if abs(row[value_col] - round(row[value_col])) < 1e-9
                     else f"{row[value_col]:,.1f}")
        common = {
            "name": row[category_col],
            "marker": {"color": color,
                       "line": {"color": marker_line_color, "width": marker_line_width}},
            "showlegend": showlegend,
            "text":         [val_label] if text_mode else None,
            "textposition": text_pos    if text_mode else None,
            "cliponaxis":   False,
        }
        if style.get("is_vertical"):
            fig.add_trace(go.Bar(x=[row[category_col]], y=[row[value_col]], **common))
        else:
            fig.add_trace(go.Bar(x=[row[value_col]], y=[row[category_col]],
                                 orientation="h", **common))

    width_px, height_px = [int(v * 110) for v in figure_size(style)]
    title_text = None
    if style.get("title_present") and title:
        title_text = title if not subtitle else f"{title}<br><sup>{subtitle}</sup>"
    title_x  = {"left": 0.01, "center": 0.5, "right": 0.99}.get(
        style.get("title_location", "center"), 0.5)
    bg       = background_color(style, rng)
    grid_x, grid_y = grid_axes(style)
    tick_color = label_color(style, rng)

    legend_dict = {"traceorder": "normal"}
    if showlegend:
        legend_dict.update(legend_location_plotly(style.get("legend_location", "right")))
        if style.get("legend_title"):
            legend_dict["title"] = {"text": category_col}
            if style.get("legend_title_color"):
                legend_dict["title"]["font"] = {"color": "#555555"}
        if style.get("legend_outline"):
            legend_dict.update({"bordercolor": "black", "borderwidth": 1,
                                 "bgcolor": "rgba(255,255,255,0.9)"})

    fig.update_layout(
        width=width_px, height=height_px,
        plot_bgcolor=bg, paper_bgcolor=bg,
        title={
            "text": title_text, "x": title_x,
            "font": {"size": title_font_size(style, rng),
                     "color": "black" if style.get("title_color") == "black" else "#555555"},
        } if title_text else None,
        showlegend=showlegend,
        legend=legend_dict,
        margin={"l": 60, "r": 40, "t": 80 if title_text else 40, "b": 60},
    )
    fig.update_xaxes(
        showgrid=grid_x, gridcolor="rgba(120,120,120,0.3)",
        tickfont={"color": tick_color},
        showticklabels=labels_visible(style.get("x_labels_orientation", "horizontal")),
        tickangle=axis_angle(style.get("x_labels_orientation", "horizontal")),
        ticks="outside" if style.get("ticks") in ["x", "both"] else "",
        showline=style.get("outline_chart"), linecolor="black",
        mirror=style.get("outline_chart"),
    )
    fig.update_yaxes(
        showgrid=grid_y, gridcolor="rgba(120,120,120,0.3)",
        tickfont={"color": tick_color},
        showticklabels=labels_visible(style.get("y_labels_orientation", "horizontal")),
        tickangle=axis_angle(style.get("y_labels_orientation", "horizontal")),
        ticks="outside" if style.get("ticks") in ["y", "both"] else "",
        showline=style.get("outline_chart"), linecolor="black",
        mirror=style.get("outline_chart"),
    )
    if style.get("direct_labeling") == "direct_no_axis":
        if style.get("is_vertical"):
            fig.update_yaxes(showticklabels=False)
        else:
            fig.update_xaxes(showticklabels=False)

    if style.get("is_vertical"):
        labels = plot_df[category_col].astype(str).tolist()
        if needs_rotation(labels, figure_size(style)[0]):
            wrapped = wrap_labels(labels)
            fig.update_xaxes(tickvals=plot_df[category_col].tolist(), ticktext=wrapped)

    return fig


def ensure_altair_available():
    global alt
    if alt is None:
        alt = ensure_module("altair", "altair")
    return alt


def ensure_altair_png_support():
    try:
        import vl_convert  # noqa: F401
    except Exception:
        install_package("vl-convert-python")


def render_bar_altair(plot_df, category_col, value_col, title, subtitle, style, rng):
    ensure_altair_available()
    grid_x, grid_y = grid_axes(style)
    lx     = label_color(style, rng)
    #scheme = SCHEME_NAMES[int(rng.integers(0, len(SCHEME_NAMES)))]
    scheme = pick_scheme(style, rng)

    labels = plot_df[category_col].astype(str).tolist()
    if style.get("is_vertical") and needs_rotation(labels, figure_size(style)[0]):
        x_axis = alt.Axis(
            labels=labels_visible(style.get("x_labels_orientation", "horizontal")),
            labelAngle=0,
            labelColor=lx, title=None, grid=grid_x,
            ticks=style.get("ticks") in ["x", "both"],
            labelExpr=r"join(split(datum.value, ' '), '\n')",
        )
    else:
        x_axis = alt.Axis(
            labels=labels_visible(style.get("x_labels_orientation", "horizontal")),
            labelAngle=axis_angle(style.get("x_labels_orientation", "horizontal"), backend="altair"),
            labelColor=lx, title=None, grid=grid_x,
            ticks=style.get("ticks") in ["x", "both"],
        )

    y_axis = alt.Axis(
        labels=labels_visible(style.get("y_labels_orientation", "horizontal")),
        labelAngle=axis_angle(style.get("y_labels_orientation", "horizontal"), backend="altair"),
        labelColor=lx, title=None, grid=grid_y,
        ticks=style.get("ticks") in ["y", "both"],
    )

    if style.get("colored_bars", True):
        legend_loc = legend_location_altair(style.get("legend_location", "no legend"))
        legend = (alt.Legend(orient=legend_loc,
                             title=category_col if style.get("legend_title") else None)
                  if legend_loc else None)
        color_encode = alt.Color(f"{category_col}:N",
                                 scale=alt.Scale(scheme=scheme), legend=legend)
    else:
        color_encode = alt.value("black")

    mark_args = {
        "cornerRadiusEnd": 4 if style.get("rounded_corners") else 0,
        "stroke":          "black" if style.get("outline_bars") else "transparent",
        "strokeWidth":     1       if style.get("outline_bars") else 0,
    }

    if style.get("is_vertical"):
        chart = alt.Chart(plot_df).mark_bar(**mark_args).encode(
            x=alt.X(f"{category_col}:N", axis=x_axis),
            y=alt.Y(f"{value_col}:Q",    axis=y_axis),
            color=color_encode,
        )
    else:
        chart = alt.Chart(plot_df).mark_bar(**mark_args).encode(
            x=alt.X(f"{value_col}:Q",    axis=x_axis),
            y=alt.Y(f"{category_col}:N", axis=y_axis),
            color=color_encode,
        )

    if style.get("direct_labeling", "none") != "none":
        if style.get("is_vertical"):
            text_mark = alt.Chart(plot_df).mark_text(
                dy=-6 if style.get("direct_label_position") == "outside" else 0
            ).encode(x=f"{category_col}:N", y=f"{value_col}:Q", text=f"{value_col}:Q")
        else:
            text_mark = alt.Chart(plot_df).mark_text(
                dx=8    if style.get("direct_label_position") == "outside" else 0,
                align="left" if style.get("direct_label_position") == "outside" else "center",
            ).encode(x=f"{value_col}:Q", y=f"{category_col}:N", text=f"{value_col}:Q")
        chart = chart + text_mark

    props = {
        "width":  int(figure_size(style)[0] * 80),
        "height": int(figure_size(style)[1] * 80),
    }
    if style.get("title_present") and title:
        title_kwargs = {
            "text":     title,
            "anchor":   title_anchor_altair(style),
            "color":    "black" if style.get("title_color") == "black" else "#555555",
            "fontSize": title_font_size(style, rng),
        }
        if subtitle:
            title_kwargs["subtitle"] = subtitle
        props["title"] = alt.TitleParams(**title_kwargs)

    chart = chart.properties(**props).configure_view(
        stroke="black" if style.get("outline_chart") else "transparent",
        strokeWidth=1,
        fill=background_color(style, rng),
    )
    if style.get("direct_labeling") == "direct_no_axis":
        if style.get("is_vertical"):
            chart = chart.configure_axisY(labels=False)
        else:
            chart = chart.configure_axisX(labels=False)
    return chart

## 7. Saving and generation


In [177]:
def ensure_output_dirs(out_root: Path) -> None:
    if CLEAR_OUTPUT and out_root.exists():
        shutil.rmtree(out_root)
    for sub in SUBDIRS:
        for lib in LIBRARIES:
            (out_root / sub / lib).mkdir(parents=True, exist_ok=True)


def chromium_executable():
    for name in ["chromium", "chromium-browser", "google-chrome", "google-chrome-stable"]:
        found = shutil.which(name)
        if found:
            return found
    return None


def save_html_screenshot(html_content: str, path: Path, width: int = 1000, height: int = 700):
    path = Path(path).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    chrome = chromium_executable()
    if chrome is None:
        raise RuntimeError("Chromium/Chrome required for HTML-to-PNG fallback.")
    with tempfile.TemporaryDirectory() as tmpdir:
        html_path = Path(tmpdir) / "chart.html"
        html_path.write_text(html_content, encoding="utf-8")
        subprocess.check_call([
            chrome, "--headless", "--disable-gpu", "--no-sandbox",
            f"--window-size={int(width)},{int(height)}",
            "--hide-scrollbars", "--force-device-scale-factor=1",
            "--virtual-time-budget=2000",
            f"--screenshot={str(path)}", html_path.as_uri(),
        ])
    return path


def save_matplotlib_png(fig, path: Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def save_plotly(fig, path: Path) -> Path:
    path = Path(path).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.write_image(str(path))
        return path
    except Exception:
        try:
            install_package("kaleido")
            fig.write_image(str(path))
            return path
        except Exception:
            html   = fig.to_html(include_plotlyjs="cdn", full_html=True)
            width  = int(fig.layout.width  or 1000)
            height = int(fig.layout.height or 700)
            return save_html_screenshot(html, path, width=width, height=height)


def save_altair(chart, path: Path) -> Path:
    path = Path(path).with_suffix(".png")
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        ensure_altair_png_support()
        chart.save(str(path))
        return path
    except Exception:
        try:
            html   = chart.to_html()
            width  = int(getattr(chart, "width",  800) or 800) + 120
            height = int(getattr(chart, "height", 500) or 500) + 120
            return save_html_screenshot(html, path, width=width, height=height)
        except Exception as exc:
            raise RuntimeError("Altair PNG export failed.") from exc


def save_metadata(meta: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)


def generate_bar(
    out_root: Path,
    dataset_source: str,
    library: str,
    rng_seed: int,
    weights: dict,
    datasets: dict | None = None,
    max_tries: int = 50,
) -> dict:
    rng      = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("bar")

    for attempt in range(1, max_tries + 1):
        style = harmonize_style(sample_style(rng, weights))
        style["_bg_color"] = background_color(style, rng)

        plot_df = None
        context = None

        if datasets:
            ds_names = list(datasets.keys())
            ds_name  = ds_names[int(rng.integers(0, len(ds_names)))]
            df       = datasets[ds_name]
            spec     = DATASET_REGISTRY[ds_name]
            result   = sample_bar_data_from_df(df, spec, rng, style)
            if result is not None:
                plot_df, context = result
                dataset_source   = ds_name
            else:
                print(f"[seed {rng_seed}] Real data sampling failed on attempt {attempt}")

        if plot_df is None:
            continue

        category_col    = context["category_col"]
        value_col       = context["value_col"]
        title, subtitle = make_title(context, style=style)

        table_path = out_root / "tables"   / library / f"{chart_id}.csv"
        meta_path  = out_root / "metadata" / library / f"{chart_id}.json"
        table_path.parent.mkdir(parents=True, exist_ok=True)
        plot_df.to_csv(table_path, index=False)

        if library == "altair":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            chart = render_bar_altair(plot_df, category_col, value_col, title, subtitle, style, rng)
            save_altair(chart, image_path)
        elif library == "matplotlib":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_bar_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng)
            save_matplotlib_png(fig, image_path)
        elif library == "seaborn":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_bar_seaborn(plot_df, category_col, value_col, title, subtitle, style, rng)
            save_matplotlib_png(fig, image_path)
        elif library == "plotly":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_bar_plotly(plot_df, category_col, value_col, title, subtitle, style, rng)
            save_plotly(fig, image_path)
        else:
            raise ValueError(f"Unsupported library: {library}")

        meta = {
            "chart_id":       chart_id,
            "chart_type":     "bar",
            "library":        library,
            "dataset_source": dataset_source,
            "image_path":     str(image_path),
            "table_path":     str(table_path),
            "data_context":   context,
            "style":          style,
            "created_utc":    datetime.utcnow().isoformat() + "Z",
            "rng_seed":       rng_seed,
        }
        save_metadata(meta, meta_path)
        return meta

    raise RuntimeError(f"Failed to generate bar after {max_tries} attempts.")


def generate_batch(
    out_root: Path,
    dataset_source: str,
    generation_plan: dict,
    weights: dict,
    datasets: dict | None = None,
    start_seed: int = 1000,
) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []
    seed  = start_seed
    for library, n in generation_plan.items():
        for _ in range(n):
            metas.append(generate_bar(
                out_root=out_root,
                dataset_source=dataset_source,
                library=library,
                rng_seed=seed,
                weights=weights,
                datasets=datasets,
            ))
            seed += 1
    return metas


## 8. Batch generation


In [178]:
ensure_output_dirs(BAR_OUT_ROOT)

OBSERVED_WEIGHTS = normalize_weights(SAMPLING_WEIGHTS)
weights = OBSERVED_WEIGHTS

generation_plan = {
    "altair":     25,
    "matplotlib": 25,
    "seaborn":    25,
    "plotly":     25,
}

metas = generate_batch(
    out_root=BAR_OUT_ROOT,
    dataset_source="warehouse_retail",
    generation_plan=generation_plan,
    weights=weights,
    datasets=DATASETS,
    start_seed=1000,
)

pd.DataFrame(metas)[["chart_id", "library", "dataset_source", "image_path"]].head()


Resorting to unclean kill browser.
Resorting to unclean kill browser.
Resorting to unclean kill browser.


,chart_id,library,dataset_source,image_path
0,bar_20260615T145853_abddd898,altair,warehouse_retail,C:\Users\Michelle\OneDrive - TU Eindhoven\I2R_...
1,bar_20260615T145857_cc8cbffd,altair,warehouse_retail,C:\Users\Michelle\OneDrive - TU Eindhoven\I2R_...
2,bar_20260615T145859_b5e51aa9,altair,london_borough_sector_jobs,C:\Users\Michelle\OneDrive - TU Eindhoven\I2R_...
3,bar_20260615T145901_353aa92c,altair,warehouse_retail,C:\Users\Michelle\OneDrive - TU Eindhoven\I2R_...
4,bar_20260615T145904_293cb1dc,altair,warehouse_retail,C:\Users\Michelle\OneDrive - TU Eindhoven\I2R_...


# TESTING

### Smoke test


In [179]:
# TEST_LIBRARIES = ["matplotlib", "seaborn", "altair", "plotly"]

# weights = normalize_weights(SAMPLING_WEIGHTS)

# # Base style: most frequent value for each parameter
# base = {param: max(probs, key=probs.get) for param, probs in weights.items()}
# base.update({
#     "title_present": True, "title_location": "center", "title_color": "black",
#     "title_size": "average", "legend_location": "right",
#     "direct_labeling": "none", "background_non_white": False,
# })

# errors = []
# rng = np.random.default_rng(42)

# for param_name, probs in weights.items():
#     for val in probs.keys():
#         style   = harmonize_style({**base, param_name: val})
#         ds_name = list(DATASETS.keys())[0]
#         df      = DATASETS[ds_name]
#         spec_ds = DATASET_REGISTRY[ds_name]
#         result  = sample_bar_data_from_df(df, spec_ds, rng, style)
#         if result is None:
#             continue
#         plot_df, context = result
#         category_col = context["category_col"]
#         value_col    = context["value_col"]
#         title, subtitle = "Test", None

#         for library in TEST_LIBRARIES:
#             try:
#                 if library == "matplotlib":
#                     fig = render_bar_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng)
#                     plt.close(fig)
#                 elif library == "seaborn":
#                     fig = render_bar_seaborn(plot_df, category_col, value_col, title, subtitle, style, rng)
#                     plt.close(fig)
#                 elif library == "altair":
#                     render_bar_altair(plot_df, category_col, value_col, title, subtitle, style, rng)
#                 elif library == "plotly":
#                     render_bar_plotly(plot_df, category_col, value_col, title, subtitle, style, rng)
#             except Exception as e:
#                 errors.append({"param": param_name, "value": val, "library": library,
#                                "error": f"{type(e).__name__}: {e}"})

# if errors:
#     print(f"Found {len(errors)} errors:")
#     pd.DataFrame(errors)
# else:
#     print("All parameter values passed for all libraries.")
